In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tempfile import mkdtemp
from joblib import Memory

# Load the dataset
data = pd.read_csv('player_valuation_dataset.csv')
data.head()

C:\Users\mtron\AppData\Local\Temp\ipykernel_37668\620166420.py:13: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('player_valuation_dataset.csv')


,player_id,player_name,date_of_birth,height,citizenship,is_eu,position,main_position,foot,current_club_id,...,goal_contributions,current_market_value,value_change_pct,total_days_injured,total_injuries,injury_proneness,club_id,competition_name,club_division,age
0,100001,Carlos Auzqui (100001),1991-03-16,180.0,Argentina,False,Attack - Right Winger,Attack,right,14554,...,79.0,200000.0,100.00,0.0,0.0,0.00,14554.0,Torneo Clausura,Torneo Clausura,34
1,1000274,Brian Romero (1000274),2006-05-11,0.0,United States Mexico,False,Attack - Right Winger,Attack,NaN,78435,...,6.0,150000.0,0.00,0.0,0.0,0.00,78435.0,Major League Soccer,Major League Soccer,19
2,1000273,Nimfasha Berchimas (1000273),2008-02-22,172.0,United States Burundi,False,Attack - Left Winger,Attack,right,78435,...,12.0,400000.0,-20.00,206.0,1.0,0.33,78435.0,Major League Soccer,Major League Soccer,18
3,1000135,Joselu Pérez (1000135),2004-03-12,183.0,Spain,True,Attack - Centre-Forward,Attack,right,8510,...,8.0,100000.0,300.00,0.0,0.0,0.00,NaN,NaN,NaN,21
4,1000284,Jed Drew (1000284),2003-08-29,176.0,Australia,False,Attack - Right Winger,Attack,right,4467,...,39.0,550000.0,266.67,40.0,3.0,1.00,4467.0,Bundesliga,Bundesliga,22


In [22]:
## Define X/y and clean data
TARGET = "current_market_value"

print(f"Initial data: {len(data)} rows")

# Check target column
print(f"Target column: {TARGET}")
print(f"Target missing: {data[TARGET].isna().sum()}")
print(f"Target infinite: {np.isinf(data[TARGET]).sum()}")

# Drop rows where target is missing or invalid
data_clean = data[data[TARGET].notna() & np.isfinite(data[TARGET])].copy()
print(f"After target cleaning: {len(data_clean)} rows")

# Replace inf/-inf with NaN in all numeric columns
numeric_cols = data_clean.select_dtypes(include=[np.number]).columns
data_clean[numeric_cols] = data_clean[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Fill NaN in numeric columns with median (skip target)
for col in numeric_cols:
    if col != TARGET and col in data_clean.columns:
        median_val = data_clean[col].median()
        if pd.isna(median_val):
            median_val = 0  # fallback if all values are NaN
        data_clean[col].fillna(median_val, inplace=True)

# Fill NaN in categorical columns with 'Unknown'
categorical_cols = data_clean.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    data_clean[col].fillna('Unknown', inplace=True)

X = data_clean.drop(columns=[TARGET])
y = data_clean[TARGET]

# Identify feature types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

C:\Users\mtron\AppData\Local\Temp\ipykernel_37668\3430467667.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_clean[col].fillna(median_val, inplace=True)
C:\Users\mtron\AppData\Local\Temp\ipykernel_37668\3430467667.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

Initial data: 33557 rows
Target column: current_market_value
Target missing: 0
Target infinite: 0
After target cleaning: 33557 rows
X shape: (33557, 40), y shape: (33557,)
NaN in X: 0
NaN in y: 0
Train size: (26845, 40)  Test size: (6712, 40)


In [23]:
# Helper to print metrics
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name:20s}  RMSE: {rmse:,.0f}  MAE: {mae:,.0f}  R²: {r2:.4f}")
    return {"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2}

# ── 6. Baseline Linear Regression ──
lr_pipe = Pipeline([("pre", preprocess), ("model", LinearRegression())])
lr_pipe.fit(X_train, y_train)

results = []
results.append(evaluate("Linear Regression", y_test, lr_pipe.predict(X_test)))

Linear Regression     RMSE: 4,065,698  MAE: 1,839,076  R²: 0.5381


c:\Users\mtron\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [24]:
# 7. Regularized models: Ridge, Lasso, Elastic Net 
for name, model in [("Ridge", Ridge()), ("Lasso", Lasso()), ("ElasticNet", ElasticNet())]:
    pipe = Pipeline([("pre", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    results.append(evaluate(name, y_test, pipe.predict(X_test)))

c:\Users\mtron\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Ridge                 RMSE: 3,917,903  MAE: 1,671,059  R²: 0.5711


c:\Users\mtron\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:656: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.980e+14, tolerance: 1.026e+14
  model = cd_fast.sparse_enet_coordinate_descent(
c:\Users\mtron\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Lasso                 RMSE: 4,325,864  MAE: 943,371  R²: 0.4771
ElasticNet            RMSE: 4,833,978  MAE: 1,513,028  R²: 0.3471


c:\Users\mtron\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
# ── 8. Hyperparameter tuning via GridSearchCV ──
X_train_pre = preprocess.fit_transform(X_train)
X_test_pre = preprocess.transform(X_test)


def tune_and_eval(name, estimator, param_grid, X_train_pre, y_train, X_test_pre, y_test, results, cv=3):
    gs = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    gs.fit(X_train_pre, y_train)
    print(f"{name} best params: {gs.best_params_}")
    results.append(evaluate(f"{name} (tuned)", y_test, gs.predict(X_test_pre)))

# Ridge
tune_and_eval(
    "Ridge",
    Ridge(),
    {"alpha": [0.1, 1, 10]},
    X_train_pre, y_train, X_test_pre, y_test, results,
    cv=3
)

# Lasso
tune_and_eval(
    "Lasso",
    Lasso(max_iter=3000, tol=1e-3, selection="random"),
    {"alpha": [0.01, 0.1, 1]},
    X_train_pre, y_train, X_test_pre, y_test, results,
    cv=3
)

# ElasticNet 
tune_and_eval(
    "ElasticNet",
    ElasticNet(max_iter=3000, tol=1e-3, selection="random"),
    {
        "alpha": [0.01, 0.1, 1],
        "l1_ratio": [0.3, 0.7]
    },
    X_train_pre, y_train, X_test_pre, y_test, results,
    cv=3
)

print("\n── Summary ──")
print(pd.DataFrame(results).sort_values("RMSE"))

c:\Users\mtron\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Ridge best params: {'alpha': 10}
Ridge (tuned)         RMSE: 3,759,902  MAE: 1,396,349  R²: 0.6050


# Next

- Advanced model RF, XGBOOST, Neural Network or SVM support vector machine
- Está a demorar bastante a correr. 